<a href="https://colab.research.google.com/github/Paras1719/GenAi-pracs/blob/main/GenAI_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Next Word Prediction using LSTM

In [ ]:
import numpy as np
import tensorflow as tf
import re
import random
import matplotlib.pyplot as plt

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

# Check GPU
if tf.config.list_physical_devices("GPU"):
    print("GPU is available")
else:
    print("GPU is NOT available")

TensorFlow version: 2.20.0
GPU is available


In [ ]:
# The dataset was previously loaded in cell JPMz-vHogVWr. We will use it here.
# Ensure `dataset` variable is available from previous execution.

In [ ]:
print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print("Testing examples:", len(dataset["test"]))

print("\nFirst conversation:")
print(dataset["train"][0])

NameError: name 'dataset' is not defined

### Data Preprocessing
First, we need to extract the text content from the dataset and clean it. Then, we will tokenize the words and create sequences for training.

In [ ]:
text = ""
for split in dataset.keys():
    for entry in dataset[split]:
        # Concatenate all utterances in a conversation, separated by a space
        text += " ".join(entry["dialog"]) + "\n"

# Convert text to lowercase and remove non-alphabetic characters
text = text.lower()
text = re.sub(r'[^a-z ]', '', text)

print("Length of text:", len(text))
# print("\nFirst 500 characters:\n", text[:500])

NameError: name 'dataset' is not defined

In [ ]:
tokenizer = Tokenizer(
    num_words=None, # Keep all words for now, will filter later if needed
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1 # +1 for OOV token

print("Vocabulary size:", total_words)

Vocabulary size: 2


In [ ]:
token_list = tokenizer.texts_to_sequences([text])[0]

print("Total tokens:", len(token_list))
print("First 30 tokens:")
print(token_list[:30])

Total tokens: 0
First 30 tokens:
[]


In [ ]:
max_sequence_len = 10 # Predict next word based on previous 9 words

input_sequences = []

for i in range(1, len(token_list)):
    sequence = token_list[max(0, i - max_sequence_len):i+1]
    if len(sequence) > 1: # Ensure sequence has at least an input and a label
        input_sequences.append(sequence)

print("Total sequences:", len(input_sequences))

Total sequences: 0


In [ ]:
input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding="pre"
)

input_sequences = np.array(
    input_sequences,
    dtype=np.int32
)

print("Input shape:", input_sequences.shape)

Input shape: (0, 10)


In [ ]:
X = input_sequences[:, :-1] # All tokens except the last one are input features
y = input_sequences[:, -1] # The last token is the label to predict

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (0, 9)
y shape: (0,)


In [ ]:
split = int(len(X) * 0.9)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 0
Testing samples: 0


### Build and Train the LSTM Model
Now, we'll define the LSTM model architecture, compile it, and train it using our prepared sequences.

In [ ]:
model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=100, # Embedding dimension
        input_length=max_sequence_len - 1 # Length of input sequences X
    ),
    LSTM(150, return_sequences=False), # LSTM layer
    Dropout(0.2), # Dropout for regularization
    Dense(
        total_words,
        activation="softmax" # Output layer with softmax for probability distribution over vocabulary
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10, # Increased epochs, but early stopping will prevent overfitting
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: 
  current = self.get_monitor_value(logs)


ValueError: math domain error

### Evaluate the Model
We will evaluate the trained model on the test set and visualize its training performance.

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("LSTM Model Accuracy")
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'history' is not defined

<Figure size 1000x600 with 0 Axes>

### Next Word Prediction Function
This function will take a seed text and predict the most likely next words using our trained LSTM model.

In [ ]:
def predict_next_words(seed_text, top_k=3):
    seed_text = seed_text.lower()

    token_list = tokenizer.texts_to_sequences(
        [seed_text]
    )[0]

    if len(token_list) == 0:
        print("No known words found in the input sentence.")
        return

    # Use only the last `max_sequence_len - 1` words as input for prediction
    token_list = token_list[-(max_sequence_len - 1):]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding="pre"
    )

    predictions = model.predict(
        token_list,
        verbose=0
    )[0]

    # Get top_k predictions
    top_indices = np.argsort(
        predictions
    )[-top_k:][::-1]

    print("\nInput:", seed_text)
    print("\nTop {top_k} suggestions:".format(top_k=top_k))

    for rank, index in enumerate(
        top_indices,
        start=1
    ):
        word = tokenizer.index_word.get(
            index, # Use the index directly
            "<UNKNOWN>"
        )

        probability = predictions[index] * 100

        print(
            f"{rank}. {word} "
            f"({probability:.2f}%)"
        )


### Demonstration of the Prediction System
Let's test the prediction function with some example phrases.

In [ ]:
predict_next_words("how are")


Input: how are

Top 3 suggestions:
1. <OOV> (50.74%)
2. <UNKNOWN> (49.26%)


In [ ]:
predict_next_words("i am feeling")


Input: i am feeling

Top 3 suggestions:
1. <OOV> (50.70%)
2. <UNKNOWN> (49.30%)


In [ ]:
predict_next_words("can you")


Input: can you

Top 3 suggestions:
1. <OOV> (50.74%)
2. <UNKNOWN> (49.26%)


### Interactive Prediction System
You can type your own sentences below to get next word predictions. Type 'exit' to stop.

In [ ]:
while True:

    sentence = input(
        "\nType a sentence (type 'exit' to stop): "
    )

    if sentence.lower() == "exit":
        print("Prediction system stopped.")
        break

    predict_next_words(sentence)


Input: i am

Top 3 suggestions:
1. <OOV> (50.74%)
2. <UNKNOWN> (49.26%)


KeyboardInterrupt: Interrupted by user